# # M-Detector: Validation of Full vs. Sparse Output Formats
# 
# **Objective:** This notebook verifies that M-Detector results saved in the `full` format (all point labels) and the `sparse_dynamic` format (indices of dynamic points) produce identical evaluation metrics.
# 
# **Methodology:**
# 1.  Load the base configuration to get default paths.
# 2.  Point to the two directories containing the `full` and `sparse` results.
# 3.  Iterate through each experiment and scene file found in the `full` results directory.
# 4.  For each scene, find its corresponding file in the `sparse` results directory.
# 5.  Dynamically locate the correct ground truth (`.pt`) file for that specific scene.
# 6.  Calculate the TP, FP, FN, and IoU metrics for both the full and sparse result files.
# 7.  Compare the results and present them in a summary table. A "PASS" indicates that the metrics are identical.

In [1]:
# %% Cell 1: Imports and Project Path Setup
# --- Standard Library Imports ---
import os
import sys
import re
import multiprocessing
from typing import Optional, Dict, Any

# --- Third-Party Imports ---
import pandas as pd
import torch
from tqdm import tqdm
import numpy as np
from IPython.display import display, HTML
from nuscenes.nuscenes import NuScenes

# --- Add Project Root to sys.path ---
# This ensures that we can import from the 'src' directory.
try:
    NOTEBOOK_DIR = os.path.dirname(__file__) # For .py script
except NameError:
    NOTEBOOK_DIR = os.path.abspath('') # For .ipynb
    
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
# Assuming the notebook is in 'm_detector_python', the project root is one level up.
# If your notebook is inside 'm_detector_python/notebooks', adjust the path.
if "m_detector_python" not in PROJECT_ROOT:
    PROJECT_ROOT = os.path.join(PROJECT_ROOT, 'm_detector_python')

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Import Custom Modules ---
# These imports will now work because the project root is in the path.
from src.config_loader import MDetectorConfigAccessor
from src.data_utils.validation_utils_torch import calculate_metrics_for_scene_from_file
from src.core.constants import OcclusionResult

# --- Configure Multiprocessing ---
# 'spawn' is safer and recommended for use with CUDA.
try:
    multiprocessing.set_start_method('spawn', force=True)
    print("Multiprocessing start method set to 'spawn'.")
except RuntimeError:
    print("Multiprocessing start method was already set.")

print("\nCell 1: Imports and Paths - OK")


Added project root to sys.path: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python
Multiprocessing start method set to 'spawn'.

Cell 1: Imports and Paths - OK


In [2]:


# %% Cell 2: Configuration, Path Definitions, and Service Loading

# --- Define Paths to Experiment Results ---
FULL_RESULTS_DIR = os.path.join(PROJECT_ROOT, "output/full")
SPARSE_RESULTS_DIR = os.path.join(PROJECT_ROOT, "output/sparse")

print("--- Experiment Data Paths ---")
print(f"Full format results path:   {FULL_RESULTS_DIR}")
print(f"Sparse format results path: {SPARSE_RESULTS_DIR}")
print("-" * 30)

# --- Load Base Configuration ---
config_accessor: Optional[MDetectorConfigAccessor] = None
GT_LABELS_BASE_DIR = ""
try:
    config_file_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    config_accessor = MDetectorConfigAccessor(config_file_path)
    nuscenes_params = config_accessor.get_nuscenes_params()
    GT_LABELS_BASE_DIR = nuscenes_params.get('label_path')
    print(f"Base configuration loaded from: {config_file_path}")
    print(f"Ground Truth labels directory set to: {GT_LABELS_BASE_DIR}")
except Exception as e:
    print(f"ERROR: Could not load base configuration. Please check the path and file. Error: {e}")

# --- Load NuScenes SDK Instance ---
nusc: Optional[NuScenes] = None
if config_accessor:
    try:
        print("\nLoading NuScenes instance... (This may take a moment)")
        nusc = NuScenes(version=nuscenes_params['version'], dataroot=nuscenes_params['dataroot'], verbose=False)
        print("NuScenes instance loaded successfully.")
    except Exception as e:
        print(f"ERROR: Failed to load NuScenes instance. Cannot proceed. Error: {e}")
else:
    print("WARNING: Cannot load NuScenes instance because base config failed to load.")

# --- Define Base Evaluation Parameters ---
base_eval_params = {
    "gt_velocity_threshold": 1.0,
    "mdet_dynamic_label_value": OcclusionResult.OCCLUDING_IMAGE.value,
}
print("\n--- Base Evaluation Parameters ---")
for key, value in base_eval_params.items():
    print(f"  {key}: {value}")
print("-" * 30)

print("\nCell 2: Configuration - OK")



--- Experiment Data Paths ---
Full format results path:   /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/output/full
Sparse format results path: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/output/sparse
------------------------------
Base configuration loaded from: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/config/m_detector_config.yaml
Ground Truth labels directory set to: /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated

Loading NuScenes instance... (This may take a moment)
NuScenes instance loaded successfully.

--- Base Evaluation Parameters ---
  gt_velocity_threshold: 1.0
  mdet_dynamic_label_value: 0
------------------------------

Cell 2: Configuration - OK


In [3]:
# %% Cell 3: Core Validation Logic - Compare Full vs. Sparse Metrics (Corrected)

validation_results = []
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f"Starting validation using device: {device}")

if nusc and os.path.isdir(FULL_RESULTS_DIR) and os.path.isdir(SPARSE_RESULTS_DIR):
    tuning_folders = sorted([d for d in os.listdir(FULL_RESULTS_DIR) if os.path.isdir(os.path.join(FULL_RESULTS_DIR, d))])
    
    print(f"Found {len(tuning_folders)} tuning experiments to compare.")

    for tuning_name in tqdm(tuning_folders, desc="Processing Tunings"):
        full_tuning_path = os.path.join(FULL_RESULTS_DIR, tuning_name)
        sparse_tuning_path = os.path.join(SPARSE_RESULTS_DIR, tuning_name)

        if not os.path.isdir(sparse_tuning_path):
            continue

        experiment_config_path = os.path.join(full_tuning_path, f"config_tuned_{tuning_name}.yaml")
        if not os.path.exists(experiment_config_path):
            continue
        exp_config_accessor = MDetectorConfigAccessor(experiment_config_path)

        proc_settings = exp_config_accessor.get_processing_settings()
        skip_frames = proc_settings.get('skip_frames')
        max_frames = proc_settings.get('max_frames') # Use -1 as a default for 'all'
        
        init_phase_settings = exp_config_accessor.get_initialization_phase_params()
        init_sweeps = init_phase_settings.get('num_sweeps_for_initial_map')

        scene_files = sorted([f for f in os.listdir(full_tuning_path) if f.startswith("mdet_results_") and f.endswith(".pt")])

        for scene_file in scene_files:
            full_pt_path = os.path.join(full_tuning_path, scene_file)
            sparse_pt_path = os.path.join(sparse_tuning_path, scene_file)

            if not os.path.exists(sparse_pt_path):
                validation_results.append({"Tuning": tuning_name, "Scene File": scene_file, "Status": "ERROR", "Reason": "Matching sparse file not found."})
                continue

            try:
                match = re.search(r'_scene_(\d+)\.pt$', scene_file)
                if not match: raise ValueError("Could not parse scene_idx from filename.")
                scene_idx = int(match.group(1))
                
                scene_name = nusc.scene[scene_idx]['name']
                
                filt_cfg = exp_config_accessor.get_point_pre_filtering_params()
                min_r, max_r = filt_cfg['min_range_meters'], filt_cfg['max_range_meters']
                
                gt_filename = f"gt_point_labels_{scene_name}_r{min_r}-{max_r}.pt"
                gt_scene_pt_path = os.path.join(GT_LABELS_BASE_DIR, gt_filename)

                if not os.path.exists(gt_scene_pt_path):
                    raise FileNotFoundError(f"Required GT file not found: {gt_scene_pt_path}")

                # --- Calculate metrics, now passing the correct slicing parameters ---
                metrics_full = calculate_metrics_for_scene_from_file(
                    gt_scene_pt_path=gt_scene_pt_path, mdet_scene_pt_path=full_pt_path,
                    eval_params=base_eval_params, device=device,
                    skip_frames=skip_frames, max_frames=max_frames, num_sweeps_for_initial_map=init_sweeps
                )
                
                metrics_sparse = calculate_metrics_for_scene_from_file(
                    gt_scene_pt_path=gt_scene_pt_path, mdet_scene_pt_path=sparse_pt_path,
                    eval_params=base_eval_params, device=device,
                    skip_frames=skip_frames, max_frames=max_frames, num_sweeps_for_initial_map=init_sweeps
                )

                if "error" in metrics_full or "error" in metrics_sparse:
                    raise RuntimeError(f"Full err: {metrics_full.get('error')}, Sparse err: {metrics_sparse.get('error')}")

                iou_full = metrics_full['tp'] / (metrics_full['tp'] + metrics_full['fp'] + metrics_full['fn']) if (metrics_full['tp'] + metrics_full['fp'] + metrics_full['fn']) > 0 else 0
                iou_sparse = metrics_sparse['tp'] / (metrics_sparse['tp'] + metrics_sparse['fp'] + metrics_sparse['fn']) if (metrics_sparse['tp'] + metrics_sparse['fp'] + metrics_sparse['fn']) > 0 else 0

                if np.isclose(iou_full, iou_sparse) and metrics_full['tp'] == metrics_sparse['tp'] and metrics_full['fp'] == metrics_sparse['fp'] and metrics_full['fn'] == metrics_sparse['fn']:
                    status, reason = "PASS", f"IoU identical ({iou_full:.4f})"
                else:
                    status, reason = "FAIL", f"Mismatch! IoU Full={iou_full:.4f}, Sparse={iou_sparse:.4f}. TP/FP/FN Full=({metrics_full['tp']}/{metrics_full['fp']}/{metrics_full['fn']}), Sparse=({metrics_sparse['tp']}/{metrics_sparse['fp']}/{metrics_sparse['fn']})"
                
                validation_results.append({"Tuning": tuning_name, "Scene File": scene_file, "Status": status, "Reason": reason})

            except Exception as e:
                validation_results.append({"Tuning": tuning_name, "Scene File": scene_file, "Status": "ERROR", "Reason": str(e)})
else:
    print("Prerequisites not met (nusc object or result directories missing). Skipping validation logic.")

print("\nCell 3: Core Validation Logic - OK")

Starting validation using device: cuda:0
Found 5 tuning experiments to compare.


Processing Tunings: 100%|█████████████████████████| 5/5 [04:07<00:00, 49.60s/it]


Cell 3: Core Validation Logic - OK


In [4]:

# %% Cell 4: Display Validation Results

if validation_results:
    df_val = pd.DataFrame(validation_results)
    
    def style_status(val):
        if val == 'PASS':
            color = 'green'
        elif val == 'FAIL':
            color = 'orange'
        else: # ERROR
            color = 'red'
        return f'color: {color}; font-weight: bold;'
    
    styled_df = df_val.style.applymap(style_status, subset=['Status']).set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    
    print("\n--- Validation Summary ---")
    display(HTML(styled_df.to_html()))
else:
    print("\nNo results to display. Please ensure your paths in Cell 2 are correct and that result files exist.")

print("\nCell 4: Display Results - OK")


--- Validation Summary ---


/tmp/ipykernel_3583204/2827632564.py:15: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_df = df_val.style.applymap(style_status, subset=['Status']).set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])])


,Tuning,Scene File,Status,Reason
0,AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p03,mdet_results_AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p03_scene_1.pt,PASS,IoU identical (0.2316)
1,AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p03,mdet_results_AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p03_scene_2.pt,PASS,IoU identical (0.3371)
2,AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p07,mdet_results_AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p07_scene_1.pt,PASS,IoU identical (0.2316)
3,AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p07,mdet_results_AdapOcc_dthr3p0_kthr0p005_dmax0p3_dmin0p07_scene_2.pt,PASS,IoU identical (0.3371)
4,Occ_BaseEps_0p07,mdet_results_Occ_BaseEps_0p07_scene_1.pt,PASS,IoU identical (0.2180)
5,Occ_BaseEps_0p07,mdet_results_Occ_BaseEps_0p07_scene_2.pt,PASS,IoU identical (0.3245)
6,Occ_BaseEps_0p13,mdet_results_Occ_BaseEps_0p13_scene_1.pt,PASS,IoU identical (0.2448)
7,Occ_BaseEps_0p13,mdet_results_Occ_BaseEps_0p13_scene_2.pt,PASS,IoU identical (0.3433)
8,ref_mccFwdDmin0p1,mdet_results_ref_mccFwdDmin0p1_scene_1.pt,PASS,IoU identical (0.2317)
9,ref_mccFwdDmin0p1,mdet_results_ref_mccFwdDmin0p1_scene_2.pt,PASS,IoU identical (0.3384)



Cell 4: Display Results - OK
